# C4-classical-ml-practice — Practice p14 — Solution

In [ ]:
import numpy as np
import pandas as pd

SEED = 20260804
FEATURES = ["length_mm", "width_mm", "mass_g", "moisture_pct"]
beans = pd.read_csv("data/beans.csv")
X = beans[FEATURES].to_numpy(dtype=float)
y = beans["species"].to_numpy()
perm = np.random.default_rng(SEED).permutation(90)
X_tr, y_tr = X[perm[:70]], y[perm[:70]]
X_te, y_te = X[perm[70:]], y[perm[70:]]
mu = X_tr.mean(axis=0)
sd = X_tr.std(axis=0)
Z_tr, Z_te = (X_tr - mu) / sd, (X_te - mu) / sd
d2_scaled = ((Z_tr[None, :, :] - Z_te[:, None, :]) ** 2).sum(axis=2)
d2_raw = ((X_tr[None, :, :] - X_te[:, None, :]) ** 2).sum(axis=2)
scaled_neighbor_labels = y_tr[np.argsort(d2_scaled, axis=1)[:, :5]]
raw_neighbor_labels = y_tr[np.argsort(d2_raw, axis=1)[:, :5]]
scaled_votes, raw_votes = [], []
for row_index in range(20):
    scaled_values, scaled_counts = np.unique(scaled_neighbor_labels[row_index], return_counts=True)
    raw_values, raw_counts = np.unique(raw_neighbor_labels[row_index], return_counts=True)
    scaled_votes.append(scaled_values[np.argmax(scaled_counts)])
    raw_votes.append(raw_values[np.argmax(raw_counts)])
preds = np.array(scaled_votes)
preds_raw = np.array(raw_votes)
test_acc = float((preds == y_te).mean())
test_acc_raw = float((preds_raw == y_te).mean())

test_acc, test_acc_raw, preds

The seeded permutation produces the specified 70/20 split, and only training means and population standard deviations transform both partitions. Standardized 5-NN scores 0.95 versus 0.90 on raw features; `length_mm` has the largest numerical spread and therefore receives disproportionate influence in the raw distances.

### Answer check

In [ ]:
expected_preds = np.array(["cava", "cava", "cava", "cava", "alba", "cava", "alba", "cava", "cava", "brio", "alba", "cava", "alba", "alba", "cava", "brio", "brio", "alba", "brio", "cava"])
assert X.shape == (90, 4) and y.shape == (90,)
assert Z_tr.shape == (70, 4) and Z_te.shape == (20, 4)
assert preds.shape == (20,) and np.array_equal(preds, expected_preds)
assert np.isclose(test_acc, 0.95, atol=1e-9, rtol=0)
assert np.isclose(test_acc_raw, 0.90, atol=1e-9, rtol=0) and test_acc > test_acc_raw